# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashiba713/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will start with **Logistic Regression** because the target is an observed yes/no outcome: whether a page is in the declining group.

Logistic Regression is a suitable first learned model because it is readable and provides a probability score for each page.

Our actual decision is a **ranking problem**: which pages should a reviewer open first? Therefore, the predicted probability of decline will be used as the ranking score, and the ranking will be evaluated using **Precision@K**, with **Precision@50** as the main metric.

I will also test a **Random Forest** as a more flexible model to check whether nonlinear relationships and interactions provide useful evidence beyond Logistic Regression.

The learned models will be compared against the frozen Week-4 baseline using the same rows, grouped split, review depths, and evaluation metrics.

The goal is decision support for content review prioritization. The models do not predict Google's ranking algorithm and do not prove that refreshing a page will improve its performance.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

The deployment question is whether the ranking approach can work for a **client the model has not seen before**.

Therefore, I use **5-fold GroupKFold by `client_id`**. Each client stays entirely inside either the training fold or the test fold.

This avoids allowing pages from the same client to appear on both sides of the split. A random row split could allow the model to learn client-specific patterns and make the evaluation look easier than the real decision.

The baseline is evaluated on the same held-out rows as each model. The baseline itself is frozen before the model comparison.

The main evaluation metric is **Precision@50**, because 50 pages represents the review depth used for the decision. Precision@10, Precision@20, and Precision@100 are also reported to show how the ranking behaves at different review depths.

A fixed random seed is used wherever randomness is involved.

In [20]:
# Split design and reproducibility setup

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold

SEED = 42
N_SPLITS = 5

np.random.seed(SEED)

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique clients:", df["client_id"].nunique())

# Observed binary target.
# 1 = observed declining, 0 = not observed declining.
df["target_decline"] = (df["trend_direction"] == "down").astype(int)

print("\nTarget distribution:")
print(df["target_decline"].value_counts())
print("\nBase rate:", round(df["target_decline"].mean(), 4))

# Check that the known label-derived fields are not being used as features.
forbidden = [
    "trend_direction",
    "trend_pct",
    "target_decline",
    "impression_change_pct"
]

print("\nForbidden fields present in raw data:")
print([c for c in forbidden if c in df.columns])

Rows: 30000
Columns: 44
Unique clients: 32

Target distribution:
target_decline
1    16262
0    13738
Name: count, dtype: int64

Base rate: 0.5421

Forbidden fields present in raw data:
['trend_direction', 'trend_pct', 'target_decline']


In [21]:
# Features available for modeling.
#
# We deliberately exclude:
# - content_id / client_id: identifiers or grouping keys
# - trend_direction / trend_pct: label-derived
# - impression_change_pct: rejected leakage feature
# - recent-vs-previous impression/click fields: excluded because
#   the previous audit showed that the impression change reproduces trend_pct
# - categorical tier columns: redundant representations of numeric fields

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions"
]

missing_features = [
    c for c in feature_columns
    if c not in df.columns
]

print("Missing requested features:", missing_features)

assert len(missing_features) == 0, (
    f"Missing features: {missing_features}"
)

X = df[feature_columns].copy()
y = df["target_decline"].copy()
groups = df["client_id"].copy()

print("\nNumber of model features:", len(feature_columns))
print("Features:")
print(feature_columns)

print("\nMissing values by feature:")
print(X.isna().sum())

Missing requested features: []

Number of model features: 14
Features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'days_with_impressions', 'days_with_sessions']

Missing values by feature:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
content_age_days             0
days_since_last_update       0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
ai_traffic_pct               0
days_with_impressions        0
days_with_sessions           0
dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Numeric preprocessing:
# median imputation handles missing values,
# StandardScaler is useful for Logistic Regression.

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", numeric_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=SEED
            )
        )
    ]
)

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median"))
                ]
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

print("Models ready:")
for name in models:
    print("-", name)

Models ready:
- Logistic Regression
- Random Forest


In [23]:
def precision_at_k(y_true, scores, k):
    """
    Precision among the top-k rows ranked by score.
    """
    k = min(k, len(y_true))

    order = np.argsort(-np.asarray(scores), kind="mergesort")
    top_k = np.asarray(y_true)[order[:k]]

    return float(np.mean(top_k))


def frozen_baseline_score(frame):
    """
    Reproduce the frozen ML-07 baseline.

    2 = stale + visible
    1 = stale + lower visibility
    0 = not yet stale

    The ML-07 visibility threshold was 139.0.
    """
    is_stale = (
        frame["days_since_last_update"] >= 90
    ).astype(int)

    is_visible = (
        frame["impressions_last_30d"] >= 139.0
    ).astype(int)

    return (
        2 * is_stale * is_visible
        + 1 * is_stale * (1 - is_visible)
    ).astype(float)


def rank_with_tie_policy(frame, scores):
    """
    Rank by model/baseline score.

    Exact score ties are broken by recent impressions,
    matching the spirit of the frozen Week-4 queue,
    with content_id providing a deterministic final tie-break.
    """
    ranked = frame[
        ["content_id", "impressions_last_30d"]
    ].copy()

    ranked["ranking_score"] = np.asarray(scores)

    ranked = ranked.sort_values(
        by=[
            "ranking_score",
            "impressions_last_30d",
            "content_id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    ).reset_index(drop=True)

    return ranked

In [24]:
# Diagnose missing values before grouped validation

print("Missing values in X:")
print(X.isna().sum())
print("\nTotal missing feature values:", X.isna().sum().sum())

print("\nMissing values in y:")
print(y.isna().sum())

print("\nMissing values in groups (client_id):")
print(groups.isna().sum())

print("\nRows with missing client_id:")
print(df.loc[groups.isna(), ["content_id", "client_id"]].head())

Missing values in X:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
content_age_days             0
days_since_last_update       0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
ai_traffic_pct               0
days_with_impressions        0
days_with_sessions           0
dtype: int64

Total missing feature values: 22927

Missing values in y:
0

Missing values in groups (client_id):
0

Rows with missing client_id:
Empty DataFrame
Columns: [content_id, client_id]
Index: []


In [25]:
# Run the complete grouped evaluation.
#
# IMPORTANT:
# Every model and the frozen baseline are evaluated on the
# exact same held-out rows in every fold.

gkf = GroupKFold(n_splits=N_SPLITS)

fold_rows = []
fold_predictions = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]

    # Hard check: no client overlap.
    overlap = set(train_groups).intersection(
        set(test_groups)
    )

    assert len(overlap) == 0, (
        f"Client leakage detected in fold {fold}: {overlap}"
    )

    # -------------------------
    # Frozen baseline
    # -------------------------
    baseline_scores = frozen_baseline_score(
        df.iloc[test_idx]
    )

    # -------------------------
    # Learned models
    # -------------------------
    model_scores = {}

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(
            X_test
        )[:, 1]

        model_scores[model_name] = probabilities

    # -------------------------
    # Evaluate everyone
    # -------------------------
    all_scores = {
        "Week-4 baseline": baseline_scores,
        **model_scores
    }

    for method_name, scores in all_scores.items():

        row = {
            "fold": fold,
            "method": method_name,
            "n_test": len(test_idx),
            "base_rate": float(y_test.mean()),
            "precision@10": precision_at_k(
                y_test, scores, 10
            ),
            "precision@20": precision_at_k(
                y_test, scores, 20
            ),
            "precision@50": precision_at_k(
                y_test, scores, 50
            ),
            "precision@100": precision_at_k(
                y_test, scores, 100
            )
        }

        fold_rows.append(row)

        # Keep row-level predictions for later error analysis.
        fold_prediction = df.iloc[test_idx][
            [
                "content_id",
                "client_id",
                "trend_direction",
                "target_decline"
            ]
        ].copy()

        fold_prediction["fold"] = fold
        fold_prediction["method"] = method_name
        fold_prediction["score"] = np.asarray(scores)

        fold_predictions.append(fold_prediction)

results_by_fold = pd.DataFrame(fold_rows)

predictions = pd.concat(
    fold_predictions,
    ignore_index=True
)

print("Grouped evaluation completed.")
print("\nClient-overlap checks passed for all folds.")

Grouped evaluation completed.

Client-overlap checks passed for all folds.


In [26]:
# Final comparison table.
#
# The base rate is shown because it tells us what random/unchosen
# pages would look like on average.

comparison_table = (
    results_by_fold
    .groupby("method")
    [
        [
            "base_rate",
            "precision@10",
            "precision@20",
            "precision@50",
            "precision@100"
        ]
    ]
    .mean()
    .reset_index()
)

comparison_table[
    [
        "base_rate",
        "precision@10",
        "precision@20",
        "precision@50",
        "precision@100"
    ]
] = comparison_table[
    [
        "base_rate",
        "precision@10",
        "precision@20",
        "precision@50",
        "precision@100"
    ]
].round(4)

comparison_table

,method,base_rate,precision@10,precision@20,precision@50,precision@100
0,Logistic Regression,0.5444,0.86,0.77,0.740,0.726
1,Random Forest,0.5444,0.66,0.71,0.684,0.664
2,Week-4 baseline,0.5444,0.62,0.51,0.492,0.500


In [27]:
print("Fold-by-fold Precision@50:")
print()

fold_p50 = (
    results_by_fold[
        [
            "fold",
            "method",
            "base_rate",
            "precision@50"
        ]
    ]
    .sort_values(
        by=["fold", "method"]
    )
    .reset_index(drop=True)
)

fold_p50

Fold-by-fold Precision@50:



,fold,method,base_rate,precision@50
0,1,Logistic Regression,0.490154,0.70
1,1,Random Forest,0.490154,0.86
2,1,Week-4 baseline,0.490154,0.42
3,2,Logistic Regression,0.645437,0.64
4,2,Random Forest,0.645437,0.86
5,2,Week-4 baseline,0.645437,0.52
6,3,Logistic Regression,0.379454,0.80
7,3,Random Forest,0.379454,0.70
8,3,Week-4 baseline,0.379454,0.42
9,4,Logistic Regression,0.622242,0.66


In [28]:
# Compare each learned model directly with the frozen baseline
# on Precision@50.

baseline_p50 = (
    results_by_fold[
        results_by_fold["method"] == "Week-4 baseline"
    ]
    .groupby("fold")["precision@50"]
    .mean()
)

model_p50 = (
    results_by_fold[
        results_by_fold["method"] != "Week-4 baseline"
    ]
    .pivot(
        index="fold",
        columns="method",
        values="precision@50"
    )
)

p50_deltas = model_p50.subtract(
    baseline_p50,
    axis=0
)

print("Precision@50 improvement over frozen baseline")
print("(positive = model better, negative = baseline better):")

p50_deltas.round(4)

Precision@50 improvement over frozen baseline
(positive = model better, negative = baseline better):


method,Logistic Regression,Random Forest
fold,,
1,0.28,0.44
2,0.12,0.34
3,0.38,0.28
4,-0.08,-0.18
5,0.54,0.08


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
## Error analysis

A high ranking score does not automatically mean that a page will decline. I therefore inspect wrong cases before interpreting the model scores.

The target is an observed directional label, not a business outcome. A false positive means the model ranked a page highly but the observed label was not declining. A false negative means the model ranked a page lower even though the observed label was declining.

I will also inspect feature importance to understand what the learned models are using. Feature importance is treated as model interpretation, not as evidence of causal relationships.

In [29]:
# Focus error analysis on the Random Forest because it is the
# more flexible model in our comparison.

rf_predictions = predictions[
    predictions["method"] == "Random Forest"
].copy()

rf_predictions["predicted_high"] = (
    rf_predictions["score"] >=
    rf_predictions.groupby("fold")["score"]
    .transform(
        lambda s: s.nlargest(min(50, len(s))).min()
    )
).astype(int)

rf_predictions["is_error"] = (
    rf_predictions["predicted_high"]
    != rf_predictions["target_decline"]
)

# Three high-scoring false positives:
false_positives = (
    rf_predictions[
        (rf_predictions["predicted_high"] == 1)
        & (rf_predictions["target_decline"] == 0)
    ]
    .sort_values(
        by="score",
        ascending=False
    )
    .head(3)
)

print("Three high-scoring false positives:")
false_positives[
    [
        "fold",
        "content_id",
        "client_id",
        "score",
        "target_decline"
    ]
]

Three high-scoring false positives:


,fold,content_id,client_id,score,target_decline
72178,4,content_9c128be31943,client_d029fa3a95,0.868598,0
69968,4,content_f79387f83703,client_d029fa3a95,0.868067,0
71169,4,content_2dbab51b83c9,client_d029fa3a95,0.866704,0


In [30]:
# Three observed declines that received relatively low model scores.

false_negatives = (
    rf_predictions[
        (rf_predictions["predicted_high"] == 0)
        & (rf_predictions["target_decline"] == 1)
    ]
    .sort_values(
        by="score",
        ascending=True
    )
    .head(3)
)

print("Three missed observed declines:")
false_negatives[
    [
        "fold",
        "content_id",
        "client_id",
        "score",
        "target_decline"
    ]
]

Three missed observed declines:


,fold,content_id,client_id,score,target_decline
70611,4,content_d19d9bb94617,client_e29c9c180c,0.065842,1
71676,4,content_31c8f34527e2,client_e29c9c180c,0.087545,1
87770,5,content_24796d98b025,client_9f14025af0,0.089380,1


In [31]:
# Build a clean Top-50 review set for Random Forest.

rf_top50 = []

for fold in sorted(
    rf_predictions["fold"].unique()
):
    fold_data = rf_predictions[
        rf_predictions["fold"] == fold
    ].copy()

    fold_top50 = (
        fold_data
        .sort_values(
            by=["score", "content_id"],
            ascending=[False, True]
        )
        .head(50)
        .copy()
    )

    fold_top50["rank"] = np.arange(
        1,
        len(fold_top50) + 1
    )

    rf_top50.append(fold_top50)

rf_top50 = pd.concat(
    rf_top50,
    ignore_index=True
)

rf_top50["error_type"] = np.select(
    [
        rf_top50["target_decline"] == 1,
        rf_top50["target_decline"] == 0
    ],
    [
        "correct_decline",
        "false_positive"
    ],
    default="unknown"
)

print("Random Forest Top-50 rows:")
print(len(rf_top50))

print("\nFalse positives in Top-50:")
print(
    rf_top50[
        rf_top50["error_type"] == "false_positive"
    ][
        [
            "fold",
            "rank",
            "content_id",
            "score",
            "target_decline"
        ]
    ].head(10)
)

Random Forest Top-50 rows:
250

False positives in Top-50:
    fold  rank            content_id     score  target_decline
29     1    30  content_433a9454796d  0.758256               0
34     1    35  content_e043a9093e34  0.757378               0
37     1    38  content_373798e89c38  0.757257               0
39     1    40  content_b7bd17e0cf9b  0.757118               0
43     1    44  content_435b4c623a4c  0.756499               0
44     1    45  content_0dd93dca6154  0.756424               0
47     1    48  content_c63693415125  0.755920               0
50     2     1  content_3bb23afaf34e  0.817472               0
75     2    26  content_be4603583853  0.771724               0
76     2    27  content_cc26e2ecb77f  0.771297               0


In [34]:
# Fit Logistic Regression once on the full dataset ONLY for
# interpretation after the grouped evaluation has already been
# completed.
#
# This interpretation is descriptive. It is NOT used to report
# the held-out evaluation score.

logistic_for_interpretation = models[
    "Logistic Regression"
]

logistic_for_interpretation.fit(
    X,
    y
)

logistic_coefficients = (
    logistic_for_interpretation
    .named_steps["model"]
    .coef_[0]
)

logistic_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "coefficient": logistic_coefficients,
            "absolute_coefficient": np.abs(
                logistic_coefficients
            )
        }
    )
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top Logistic Regression features:")
logistic_importance.head(10)

Top Logistic Regression features:


,feature,coefficient,absolute_coefficient
0,days_with_impressions,0.593429,0.593429
1,days_with_sessions,-0.387652,0.387652
2,content_age_days,-0.387269,0.387269
3,word_count,0.347384,0.347384
4,char_count,-0.257991,0.257991
5,days_since_last_update,0.124497,0.124497
6,avg_position,-0.120622,0.120622
7,scroll_rate,0.099339,0.099339
8,ctr,-0.076813,0.076813
9,engagement_rate,-0.040901,0.040901


In [33]:
# Fit Random Forest once on the full dataset ONLY for
# interpretation after the grouped evaluation.

rf_for_interpretation = models[
    "Random Forest"
]

rf_for_interpretation.fit(
    X,
    y
)

rf_importances = (
    rf_for_interpretation
    .named_steps["model"]
    .feature_importances_
)

rf_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": rf_importances
        }
    )
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top Random Forest features:")
rf_importance.head(10)

Top Random Forest features:


,feature,importance
0,days_with_impressions,0.283077
1,avg_position,0.190562
2,content_age_days,0.159650
3,char_count,0.063971
4,word_count,0.057350
5,days_with_sessions,0.048662
6,days_since_last_update,0.045220
7,ctr,0.045092
8,search_volume,0.038361
9,scroll_rate,0.034601


## Interpretation

The learned models are being used for ranking rather than as a claim about causation.

The most important features are interpreted as signals that the fitted model relied on, not as causes of decline. In particular, a feature can be predictive because it is associated with the observed label without causing the outcome.

The grouped evaluation is the main evidence. I compare the models with the frozen Week-4 baseline on the same held-out client groups and use Precision@50 as the primary decision metric.

If the Random Forest has a higher average Precision@50 but the improvement is inconsistent across client folds, I will describe the result as evidence of some ranking signal with instability rather than declaring the model a stable winner.

If a simpler model performs similarly to the Random Forest, the simpler model is preferable because it is easier to understand and maintain.

The error cases also show why the ranking should be treated as decision support. Some highly ranked pages may not actually decline, while some observed declines may receive low scores because their available features resemble healthier pages.

These results do not prove that refreshing a page causes improvement, and they do not predict Google's ranking algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.